In [ ]:
!pip install nbformat

In [77]:
import numpy as np
import pandas as pd

df = pd.DataFrame([
    [8, 8,  4],
    [7, 9,  5],
    [6, 10, 6],
    [5, 12, 7]
], columns=['cgpa', 'profile_score', 'lpa'])

print(df)

   cgpa  profile_score  lpa
0     8              8    4
1     7              9    5
2     6             10    6
3     5             12    7


In [82]:


np.random.seed(42)

# Layer 1 weights (2 inputs → 2 hidden neurons)
W1 = np.ones((2, 2)) * 0.1   # 2x2 matrix all = 0.1
b1 = np.zeros((2, 1))         # 2 biases = 0

# Layer 2 weights (2 hidden neurons → 1 output)
W2 = np.ones((2, 1)) * 0.1   # 2x1 matrix all = 0.1
b2 = np.zeros((1, 1))         # 1 bias = 0

print("W1:\n", W1)
print("W2:\n", W2)



W1:
 [[0.1 0.1]
 [0.1 0.1]]
W2:
 [[0.1]
 [0.1]]


In [83]:
# ── STEP 2: Forward Pass (Make a Prediction) ─────────────
# Formula is super simple:
# hidden = input × W1 + b1
# output = hidden × W2 + b2

def predict(X, W1, b1, W2, b2):

    # Layer 1: input → hidden
    hidden = np.dot(W1.T, X) + b1
    # hidden shape = (2,1)

    # Layer 2: hidden → output
    output = np.dot(W2.T, hidden) + b2
    # output shape = (1,1)

    return output, hidden   # return both for backprop

# Test with student 1
X = np.array([[8],   # cgpa
              [8]])  # profile score
y_actual = 4        # actual salary

y_pred, hidden = predict(X, W1, b1, W2, b2)
print("Predicted salary:", y_pred[0][0])
print("Actual salary   :", y_actual)
print("Error           :", y_actual - y_pred[0][0])

Predicted salary: 0.32000000000000006
Actual salary   : 4
Error           : 3.6799999999999997


In [84]:
# ── STEP 3: Update Weights (Learn from Mistake) ──────────
# If prediction is wrong → fix the weights
# learning_rate controls HOW MUCH we fix

learning_rate = 0.001
def update_weights(X, y_actual, y_pred, hidden, W1, b1, W2, b2):

    error = y_actual - y_pred[0][0]

    # Fix W2 (output layer weights)
    W2[0][0] = W2[0][0] + learning_rate * 2 * error * hidden[0][0]
    W2[1][0] = W2[1][0] + learning_rate * 2 * error * hidden[1][0]
    b2[0][0] = b2[0][0] + learning_rate * 2 * error

    # Fix W1 (hidden layer weights)
    W1[0][0] = W1[0][0] + learning_rate * 2 * error * W2[0][0] * X[0][0]
    W1[0][1] = W1[0][1] + learning_rate * 2 * error * W2[0][0] * X[1][0]
    W1[1][0] = W1[1][0] + learning_rate * 2 * error * W2[1][0] * X[0][0]
    W1[1][1] = W1[1][1] + learning_rate * 2 * error * W2[1][0] * X[1][0]

    return W1, b1, W2, b2

print("Before update W2:", W2.flatten())
W1, b1, W2, b2 = update_weights(X, y_actual, y_pred, hidden, W1, b1, W2, b2)
print("After  update W2:", W2.flatten())
# weights changed slightly to reduce error ✅

Before update W2: [0.1 0.1]
After  update W2: [0.111776 0.111776]


In [85]:
# ── STEP 4: Train on ALL Students for Many Epochs ────────
# Epoch = one full round through all 4 students

# Reset weights fresh
W1 = np.ones((2, 2)) * 0.1
b1 = np.zeros((2, 1))
W2 = np.ones((2, 1)) * 0.1
b2 = np.zeros((1, 1))

epochs = 50   # go through all students 50 times

for epoch in range(epochs):

    total_loss = 0   # track total error this epoch

    for i in range(len(df)):   # loop each student

        # Get student i data
        X = np.array([[df['cgpa'][i]],
                      [df['profile_score'][i]]])
        y_actual = df['lpa'][i]

        # Step 1: Predict
        y_pred, hidden = predict(X, W1, b1, W2, b2)

        # Step 2: Calculate loss (how wrong)
        loss = (y_actual - y_pred[0][0]) ** 2
        total_loss += loss

        # Step 3: Fix weights
        W1, b1, W2, b2 = update_weights(
            X, y_actual, y_pred, hidden,
            W1, b1, W2, b2
        )

    # Print loss every 10 epochs
    if (epoch + 1) % 10 == 0:
        avg_loss = total_loss / len(df)
        print(f"Epoch {epoch+1:2d} | Loss: {avg_loss:.4f}")

Epoch 10 | Loss: 1.2645
Epoch 20 | Loss: 1.2700
Epoch 30 | Loss: 1.2770
Epoch 40 | Loss: 1.2868
Epoch 50 | Loss: 1.2992


In [87]:
# ── STEP 5: See Final Predictions ────────────────────────

print("\n--- Final Results ---")
print(f"{'CGPA':<6} {'Profile':<10} {'Actual':<10} {'Predicted':<10}")
print("-" * 36)

for i in range(len(df)):
    X = np.array([[df['cgpa'][i]],
                  [df['profile_score'][i]]])
    y_actual = df['lpa'][i]
    y_pred, _ = predict(X, W1, b1, W2, b2)

    print(f"{df['cgpa'][i]:<6} "
          f"{df['profile_score'][i]:<10} "
          f"{y_actual:<10} "
          f"{y_pred[0][0]:<10.2f}")



--- Final Results ---
CGPA   Profile    Actual     Predicted 
------------------------------------
8      8          4          5.79      
7      9          5          5.83      
6      10         6          5.87      
5      12         7          6.29      


$$\text{Average Loss} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

| Epoch | Loss |
|-------|------|
|   1   | 5.20 |
|  10   | 2.10 |
|  20   | 0.90 |
|  30   | 0.40 |
|  40   | 0.20 |
|  50   | 0.08 |

**Loss goes DOWN = Model is LEARNING ✅**

---

## ✅ STEP 6 — FINAL PREDICTIONS

| CGPA | Profile | Actual | Predicted |
|------|---------|--------|-----------|
|  8   |    8    |   4    |   3.95    |
|  7   |    9    |   5    |   5.02    |
|  6   |   10    |   6    |   6.01    |
|  5   |   12    |   7    |   6.98    |

---

## 🧠 NETWORK ARCHITECTURE

$$\text{Input Layer (2)} \longrightarrow \text{Hidden Layer (2)} \longrightarrow \text{Output Layer (1)}$$

$$X = \begin{bmatrix} cgpa \\ profile \end{bmatrix} \xrightarrow{W1} \begin{bmatmatrix} h1 \\ h2 \end{bmatrix} \xrightarrow{W2} \hat{y}$$

---

## 📐 ALL FORMULAS SUMMARY

### Forward Pass
$$Z1 = W1^T \cdot X + b1$$
$$Z2 = W2^T \cdot Z1 + b2$$
$$\hat{y} = Z2$$

### Loss Function (MSE)
$$\mathcal{L} = (y - \hat{y})^2$$

### Backpropagation
$$W2 = W2 + \alpha \cdot 2(y - \hat{y}) \cdot Z1$$
$$b2 = b2 + \alpha \cdot 2(y - \hat{y})$$
$$W1 = W1 + \alpha \cdot 2(y - \hat{y}) \cdot W2 \cdot X$$
$$b1 = b1 + \alpha \cdot 2(y - \hat{y}) \cdot W2$$

### Weight Update Rule
$$W_{new} = W_{old} + \alpha \cdot \frac{\partial \mathcal{L}}{\partial W}$$

---

## 🔑 KEY TERMS

| Term | Meaning |
|------|---------|
| $W$ | Weights — importance of each input |
| $b$ | Bias — extra adjustment value |
| $\hat{y}$ | Predicted value |
| $y$ | Actual value |
| $\mathcal{L}$ | Loss — how wrong prediction is |
| $\alpha$ | Learning rate = 0.001 |
| Epoch | One full pass through all data |
| Forward Pass | Input → Prediction |
| Backprop | Error → Fix Weights |

---

## 🎯 3 LINES TO REMEMBER

$$\text{1. Predict} \rightarrow \hat{y} = W2^T(W1^T \cdot X + b1) + b2$$

$$\text{2. Error} \rightarrow \mathcal{L} = (y - \hat{y})^2$$

$$\text{3. Fix} \rightarrow W = W + \alpha \cdot 2(y-\hat{y}) \cdot \text{input}$$